# Huấn luyện CNN bằng PyTorch

Notebook PyTorch dùng **cùng dataset** với Keras và NumPy: `dataset_raw/animals/animals`, có cấu trúc `class_name/image_file`.

In [ ]:
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 42
BATCH_SIZE = 32
DATASET_DIR = Path('dataset_raw/animals/animals')
assert DATASET_DIR.exists(), f'Không tìm thấy {DATASET_DIR.resolve()}'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '| dataset chung:', DATASET_DIR.resolve())

## 1. DataLoader của dataset chung

`ImageFolder` suy ra nhãn trực tiếp từ tên thư mục lớp. Dataset được tách train/validation 80/20 bằng seed 42, cùng quy ước với notebook Keras.

In [ ]:
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
full_dataset = datasets.ImageFolder(DATASET_DIR, transform=transform)
class_names = full_dataset.classes
train_size = int(.8 * len(full_dataset))
valid_size = len(full_dataset) - train_size
train_dataset, valid_dataset = random_split(full_dataset, [train_size, valid_size], generator=torch.Generator().manual_seed(SEED))
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
print(f'{len(class_names)} lớp | train={len(train_dataset)} | validation={len(valid_dataset)}')

## 2. Định nghĩa `nn.Module`

Conv2d + ReLU + MaxPool2d trích xuất đặc trưng. AdaptiveAvgPool2d giúp đầu classifier không phụ thuộc kích thước không gian cuối.

In [ ]:
class AnimalCNN(nn.Module):
    def __init__(self, class_count):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(128, class_count)
    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))

model = AnimalCNN(len(class_names)).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
model

## 3. Huấn luyện và lưu trọng số

`loss.backward()` thực hiện autograd; `optimizer.step()` cập nhật tham số. File `.pt` lưu state dict kèm tên lớp.

In [ ]:
EPOCHS = 15
for epoch in range(EPOCHS):
    model.train(); correct = total = 0; total_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct += (logits.argmax(1) == labels).sum().item(); total += len(labels)
    model.eval(); valid_correct = valid_total = 0
    with torch.no_grad():
        for images, labels in valid_loader:
            logits = model(images.to(DEVICE))
            valid_correct += (logits.argmax(1).cpu() == labels).sum().item(); valid_total += len(labels)
    print(f'Epoch {epoch + 1:02d}: loss={total_loss / total:.4f}, train={correct / total:.2%}, validation={valid_correct / valid_total:.2%}')

torch.save({'state_dict': model.state_dict(), 'classes': class_names}, 'animal_cnn_pytorch.pt')
print('Đã lưu animal_cnn_pytorch.pt')